# 在 Colab 中打开
<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ai-agents-the-definitive-guide/blob/main/CH07/ch07_hardening_backbone.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# 关于本 Notebook

从诊断实验室走向生产 API，意味着你要从 **"agent vibes（凭感觉做智能体）"** 进入 **"system physics（系统工程约束）"**。本 Notebook 实现了让智能体能够在生产环境中持续可靠运行的三个基础支柱：

1. **检查点（Checkpointing）** —— 周期性保存状态快照，让智能体在进程重启后可以恢复。
2. **上下文裁剪（Context Pruning）** —— 按 token 预算截断历史，避免“记忆膨胀”。
3. **熔断器（Circuit Breaker）** —— 依据轮次和成本设置保险丝，阻止失控循环。

| 模式 | 关注点 | 机制 | 为什么需要 |
|---|---|---|---|
| Checkpointing | 可靠性 | 周期性把状态快照保存到磁盘 / Redis | 如果进程在 10 步中的第 4 步崩溃，智能体可以从第 4 步恢复，而不是从头开始。 |
| Context Pruning | 性能 | 按 token 预算裁剪历史 | 防止“上下文膨胀（Context Bloat）”：陈旧且无关的消息会削弱推理质量并显著增加成本。 |
| Circuit Breaker | 安全 | 按轮次和成本设置熔断条件 | 当智能体陷入工具调用失败的无限循环，或超过预算时及时停止。 |

我们会把这三种模式组合成一个**加固后的智能体循环（hardened agent loop）**，再通过实际演示验证每一种机制都能生效。

### 1. `AgentSession` 让恢复成为可能
通过 Pydantic 把智能体的完整状态变成可序列化对象，你就可以在每一轮结束后保存快照。如果进程在 10 步中的第 4 步崩溃，恢复时可以直接从第 4 步继续，而不是从零开始。生产环境通常会把状态写入 Redis 或 Postgres，而不是 JSON 文件。

### 2. 上下文裁剪不是可选项
随着上下文持续增长，模型会变慢，也更容易失去重点。Context Janitor 会保留系统 Prompt、保留最近轮次，并对被丢弃的内容进行摘要。它不是简单的“清理”，而是一种性能与成本控制机制。

### 3. 为每个 Session 硬编码预算上限
熔断器是防御无限循环的第一道防线。一个卡在工具调用循环中的智能体，几分钟就可能耗掉大量 API 预算。为每个 Session 设置 0.50 美元的上限，是成本很低的保险措施。当熔断器触发时，FastAPI 层可以捕获异常，并返回用户友好的“我需要人工介入”提示。

### 4. 这些模式可以组合
Checkpointing、Context Pruning 和 Circuit Breaker 不是彼此独立的功能，而是同一个循环中的多层保护。每一轮都执行：裁剪 → 调用 → 更新指标 → 检查熔断器 → 保存检查点。缺少任何一层，系统都会留下盲区。

### 5. 流式传输（Streaming）与轮询（Polling）
在 FastAPI 的生产部署中，长时间运行的智能体任务应尽量避免使用普通 REST `POST` 请求。智能体需要时间“思考”，30 秒超时并不少见。可以使用 **WebSockets** 或 **服务器发送事件（Server-Sent Events，SSE）** 把智能体的执行过程持续推送到 UI。它不会让模型本身变快，但用户能够看到智能体正在工作，从而降低感知延迟。

In [ ]:
%pip install -q openai pydantic python-dotenv tiktoken

In [ ]:
import json, os, time, tempfile, shutil
from pathlib import Path
from typing import Any, Dict, List, Literal, Optional
from dataclasses import dataclass, field

from pydantic import BaseModel, Field
from dotenv import load_dotenv
from openai import OpenAI
import tiktoken

load_dotenv()

OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')
BASE = "https://openrouter.ai/api/v1"

client = OpenAI(
    base_url=BASE,
    api_key=OPENROUTER_API_KEY,
)

MODEL = "openai/gpt-4o-mini"

# 使用与模型匹配的 tiktoken 编码，以获得调用前上下文管理所需的精确 token 计数，
# 而不是教程中常见的 chars÷4 粗略估算。
ENCODING = tiktoken.encoding_for_model("gpt-4o-mini")

print(f"Model:    {MODEL}")
print(f"API base: {BASE}")
print(f"Encoder:  {ENCODING.name} ({len(ENCODING._mergeable_ranks):,} merge rules)")

Model:    openai/gpt-4o-mini
API base: https://openrouter.ai/api/v1
Encoder:  o200k_base (199,998 merge rules)


## 智能体工具（Agent Tools）

在生产环境中，这些函数可能会调用订单数据库、策略服务和工单分类器。这里使用带有真实感数据的确定性替代实现，使加固模式能够被测试并复现；但工具 Schema、Dispatcher（分发器）以及函数调用契约（function-calling contract）的结构，与真正上线时采用的方式是一致的。

In [ ]:
# 工具定义（OpenAI function-calling schema）

TOOL_SCHEMAS: list[dict] = [
    {
        "type": "function",
        "function": {
            "name": "classify_ticket",
            "description": (
                "Classify a customer support ticket into a category. "
                "Returns the category and a confidence score."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "summary": {
                        "type": "string",
                        "description": "A one-sentence summary of the customer's issue.",
                    },
                    "category": {
                        "type": "string",
                        "enum": ["billing", "technical", "account", "general"],
                        "description": "The most fitting category for this ticket.",
                    },
                },
                "required": ["summary", "category"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "lookup_order",
            "description": (
                "Look up an order by its ID and return current status, "
                "shipping estimate, and item details."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "The order identifier, e.g. ORD-1024.",
                    },
                },
                "required": ["order_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_policy",
            "description": (
                "Look up a company policy by keyword. Useful for return, "
                "refund, warranty, or shipping policy questions."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "keyword": {
                        "type": "string",
                        "description": "Policy topic to search for, e.g. 'return', 'refund', 'warranty'.",
                    },
                },
                "required": ["keyword"],
            },
        },
    },
]


# 模拟数据

_MOCK_ORDERS: dict[str, dict] = {
    "ORD-1024": {
        "status": "processing",
        "placed": "2025-02-04",
        "estimated_ship": "2025-02-14",
        "items": ["Wireless Keyboard x1", "USB-C Hub x1"],
    },
    "ORD-5567": {
        "status": "shipped",
        "placed": "2025-02-01",
        "tracking": "1Z999AA10123456784",
        "estimated_delivery": "2025-02-12",
        "items": ["Running Shoes (Size 10) x1"],
    },
    "ORD-8842": {
        "status": "delayed",
        "placed": "2025-01-20",
        "reason": "Supplier back-order; new estimated ship date 2025-02-18.",
        "items": ["Standing Desk Frame x1", "Monitor Arm x2"],
    },
}

_MOCK_POLICIES: dict[str, str] = {
    "return": (
        "Items may be returned within 30 days of delivery in original "
        "packaging. Electronics must be unopened or defective. Refunds "
        "are processed within 5-7 business days."
    ),
    "refund": (
        "Refunds are issued to the original payment method. Partial "
        "refunds may apply if the item shows signs of use. Shipping "
        "costs are non-refundable unless the return is due to our error."
    ),
    "warranty": (
        "All electronics carry a 1-year limited warranty covering "
        "manufacturing defects. Accessories are covered for 90 days. "
        "File a warranty claim through your account dashboard."
    ),
    "shipping": (
        "Standard shipping: 5-7 business days. Express: 2-3 business "
        "days. Free standard shipping on orders over $50. International "
        "shipping available to select countries."
    ),
}


# 模拟工具实现

def classify_ticket(summary: str, category: str) -> str:
    confidence = 0.92 if category in ("billing", "technical", "account") else 0.78
    return json.dumps({"category": category, "summary": summary, "confidence": confidence})


def lookup_order(order_id: str) -> str:
    order_id = order_id.strip().upper()
    if order_id in _MOCK_ORDERS:
        return json.dumps({"order_id": order_id, **_MOCK_ORDERS[order_id]})
    return json.dumps({"order_id": order_id, "error": "Order not found. Please verify the order ID."})


def check_policy(keyword: str) -> str:
    keyword = keyword.strip().lower()
    for key, text in _MOCK_POLICIES.items():
        if key in keyword or keyword in key:
            return json.dumps({"policy": key, "text": text})
    return json.dumps({"keyword": keyword, "text": "No specific policy found. Contact support@example.com."})


# 分发器（Dispatcher）

TOOL_MAP: dict[str, callable] = {
    "classify_ticket": classify_ticket,
    "lookup_order": lookup_order,
    "check_policy": check_policy,
}


def execute_tool(name: str, arguments: dict) -> str:
    """按名称分发工具调用，并返回 JSON 字符串结果。"""
    fn = TOOL_MAP[name]
    return fn(**arguments)


print(f"✅ {len(TOOL_SCHEMAS)} tools registered: {list(TOOL_MAP.keys())}")
# 快速冒烟测试
print(f"   lookup_order('ORD-1024') → {lookup_order('ORD-1024')[:60]}...")
print(f"   check_policy('return')   → {check_policy('return')[:60]}...")

✅ 3 tools registered: ['classify_ticket', 'lookup_order', 'check_policy']
   lookup_order('ORD-1024') → {"order_id": "ORD-1024", "status": "processing", "placed": "...
   check_policy('return')   → {"policy": "return", "text": "Items may be returned within 3...


# `AgentSession`：可序列化状态（Serializable State）

`AgentSession` 让智能体的完整状态可以被序列化。它就是你需要保存到 Redis、Postgres，或者本演示中的磁盘 JSON 文件里的“状态大脑”。

所有与恢复有关的重要字段都包含在这里：对话历史、轮次计数、累计成本以及 Session 状态。如果进程退出，只要重新加载这个对象，就可以从中断的位置继续执行。

In [ ]:
class AgentSession(BaseModel):
    """可序列化的智能体状态——需要保存到检查点里的“状态大脑”。"""
    session_id: str
    model: str = MODEL
    system_prompt: str = "You are a customer support triage agent."
    history: List[Dict[str, Any]] = Field(default_factory=list)
    turn_count: int = 0
    total_tokens: int = 0
    total_usd: float = 0.0
    status: Literal["running", "halted", "completed"] = "running"
    halt_reason: Optional[str] = None
    checkpoints_saved: int = 0

    def add_message(self, role: str, content: str, **extra):
        """向对话历史追加一条消息。"""
        msg = {"role": role, "content": content, **extra}
        self.history.append(msg)

    def summary(self) -> str:
        return (
            f"Session {self.session_id} | status={self.status} | "
            f"turns={self.turn_count} | tokens={self.total_tokens} | "
            f"cost=${self.total_usd:.4f} | messages={len(self.history)} | "
            f"checkpoints={self.checkpoints_saved}"
        )


# 快速健全性检查
s = AgentSession(session_id="test-001")
s.add_message("user", "Hello")
print(s.summary())
print(f"Serializes to {len(s.model_dump_json())} bytes of JSON")

Session test-001 | status=running | turns=0 | tokens=0 | cost=$0.0000 | messages=1 | checkpoints=0
Serializes to 267 bytes of JSON


# Token 计数与成本跟踪（Token Counting & Cost Tracking）

这里有两个不同任务，因此也需要两个不同的事实来源：

1. **调用前（上下文管理）：** 在发送请求之前，我们要知道历史消息会占用多少 token，Context Janitor 才能判断是否需要裁剪。这里使用 `tiktoken`——与模型一致的 BPE（Byte Pair Encoding）分词器——进行精确计数。
2. **调用后（成本跟踪）：** 每次 API 响应后，OpenRouter 会返回真实的 `usage.prompt_tokens` 和 `usage.completion_tokens`。计费应使用这些实际用量，而不是本地估算值。

In [ ]:
# 调用前：基于 tiktoken 的 token 计数
def count_message_tokens(message: Dict[str, Any]) -> int:
    """使用 tiktoken 计算单条聊天消息的 token 数量。

    遵循 OpenAI 的 token 计数方法：每条消息都有固定的
    role/name 结构开销，再加上消息内容本身的 token。
    See: https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb
    """
    # 每条消息的固定开销（role、分隔符）；gpt-4o / gpt-4o-mini 为 4 个 token
    tokens = 4
    for key, value in message.items():
        if isinstance(value, str):
            tokens += len(ENCODING.encode(value))
        elif isinstance(value, list):
            # tool_calls、repro_steps 等
            tokens += len(ENCODING.encode(json.dumps(value)))
    return tokens


def count_tokens(messages: List[Dict[str, Any]]) -> int:
    """计算完整对话历史的 token 总数。

    其中包含 API 在最后一条消息之后添加的 2-token 回复起始标记（reply primer）。
    """
    total = sum(count_message_tokens(m) for m in messages)
    total += 2  # 回复起始标记（reply primer）
    return total


# 调用后：根据 API 返回的真实用量计算成本

# gpt-4o-mini 价格（每 1K token）；价格变化时需要同步更新
COST_PER_1K_INPUT  = 0.00015   # $0.15  / 1M input tokens
COST_PER_1K_OUTPUT = 0.0006    # $0.60  / 1M output tokens


def calculate_cost(prompt_tokens: int, completion_tokens: int) -> float:
    """根据 API 返回的 token 用量计算美元成本。"""
    return (
        (prompt_tokens / 1000) * COST_PER_1K_INPUT +
        (completion_tokens / 1000) * COST_PER_1K_OUTPUT
    )


# 演示：比较精确计数与粗略估算的差异

demo_msgs = [
    {"role": "system", "content": "You are a customer support triage agent."},
    {"role": "user", "content": "I placed order #ORD-8842 three weeks ago and nobody has told me what's going on."},
]

real_count = count_tokens(demo_msgs)
naive_guess = sum(len(str(m.get("content", ""))) for m in demo_msgs) // 4

print(f"tiktoken count:  {real_count} tokens")
print(f"chars÷4 guess:   {naive_guess} tokens")
print(f"Error:           {abs(real_count - naive_guess)} tokens ({abs(real_count - naive_guess)/real_count*100:.0f}%)")
print(f"\nCost for 1000 in + 500 out: ${calculate_cost(1000, 500):.6f}")

tiktoken count:  41 tokens
chars÷4 guess:   30 tokens
Error:           11 tokens (27%)

Cost for 1000 in + 500 out: $0.000450


# Context Janitor：历史裁剪（History Pruning）

高端模型虽然拥有很大的上下文窗口，但把窗口完全塞满通常是错误做法。它会增加延迟，也会让智能体的注意力被“摊薄”。Context Janitor 通过下面三条规则保持历史精简：

1. **永远不裁剪系统 Prompt** —— 它包含智能体最核心的身份和行为约束。
2. **只保留最近的若干轮次** —— 保住最直接、最相关的上下文。
3. **为被裁剪内容注入摘要** —— 在缩短上下文的同时保留长期信息。

In [ ]:
def prune_history(
    history: List[Dict],
    max_tokens: int = 4000,
    keep_recent: int = 6,
) -> List[Dict]:
    """
    Context Janitor：保留系统 Prompt 和最近 N 轮对话，
    丢弃中间历史，并注入一条被裁剪内容的摘要。

    token 数量使用 tiktoken 计算——它采用与模型一致的 BPE（Byte Pair Encoding）分词器，
    因此裁剪阈值来自精确计数，而不是粗略猜测。
    """
    current_tokens = count_tokens(history)
    if current_tokens <= max_tokens:
        return history  # 无需裁剪

    # 永远不裁剪系统 Prompt（第一条消息）
    system_msg = history[0] if history and history[0].get("role") == "system" else None

    if system_msg:
        middle = history[1:-keep_recent] if len(history) > keep_recent + 1 else []
        recent = history[-keep_recent:]
    else:
        middle = history[:-keep_recent] if len(history) > keep_recent else []
        recent = history[-keep_recent:]

    # 为即将被裁剪的内容生成摘要
    n_pruned = len(middle)
    pruned_roles = [m.get("role", "?") for m in middle]
    role_counts = {r: pruned_roles.count(r) for r in set(pruned_roles)}
    role_summary = ", ".join(f"{count} {role}" for role, count in role_counts.items())

    summary_msg = {
        "role": "system",
        "content": (
            f"[Context Janitor] Pruned {n_pruned} older messages "
            f"({role_summary}) to stay within token budget. "
            f"Recent context preserved below."
        ),
    }

    pruned = []
    if system_msg:
        pruned.append(system_msg)
    pruned.append(summary_msg)
    pruned.extend(recent)

    after_tokens = count_tokens(pruned)
    print(
        f"  🧹 Context Janitor: {len(history)} msgs ({current_tokens:,} tokens) → "
        f"{len(pruned)} msgs ({after_tokens:,} tokens) | pruned {n_pruned} middle messages"
    )

    return pruned


# 演示：构造膨胀的历史并执行裁剪
demo_history = [{"role": "system", "content": "You are a helpful agent."}]
for i in range(20):
    demo_history.append({"role": "user", "content": f"Turn {i}: " + "x" * 200})
    demo_history.append({"role": "assistant", "content": f"Reply {i}: " + "y" * 200})

print(f"Before: {len(demo_history)} messages, {count_tokens(demo_history):,} tokens (tiktoken)")
pruned = prune_history(demo_history, max_tokens=1500, keep_recent=6)
print(f"After:  {len(pruned)} messages, {count_tokens(pruned):,} tokens (tiktoken)")
print(f"First message role: {pruned[0]['role']}")
print(f"Second message: {pruned[1]['content'][:100]}...")

Before: 41 messages, 1,933 tokens (tiktoken)
  🧹 Context Janitor: 41 msgs (1,933 tokens) → 8 msgs (336 tokens) | pruned 34 middle messages
After:  8 messages, 336 tokens (tiktoken)
First message role: system
Second message: [Context Janitor] Pruned 34 older messages (17 assistant, 17 user) to stay within token budget. Rece...


# 熔断器（Circuit Breaker）

为每个 Session 硬编码预算上限，是防止无限循环的核心保护。这里的熔断器包含两根“保险丝”：

- **预算保险丝（Budget Fuse）：** 如果 Session 的累计成本超过 `BUDGET_CAP`，立即停止。
- **轮次保险丝（Turn Fuse）：** 如果智能体的推理轮次超过 `MAX_TURNS`，立即停止。

熔断器触发后会抛出 `CircuitBreakerTripped` 异常。FastAPI 层可以捕获这个异常，并转换成用户友好的“我卡住了，需要人工介入”提示。

In [ ]:
class CircuitBreakerTripped(RuntimeError):
    """当智能体超过运行预算或轮次上限时抛出。"""
    def __init__(self, reason: str, session: AgentSession):
        self.reason = reason
        self.session = session
        super().__init__(reason)


def check_circuit_breaker(
    session: AgentSession,
    budget_cap: float = 0.50,
    max_turns: int = 12,
):
    """
    当智能体持续消耗预算或陷入循环时触发熔断。
    应在每次 LLM 调用之后执行此检查。
    """
    if session.total_usd >= budget_cap:
        session.status = "halted"
        session.halt_reason = f"Budget exceeded: ${session.total_usd:.4f} >= ${budget_cap:.2f}"
        raise CircuitBreakerTripped(session.halt_reason, session)

    if session.turn_count >= max_turns:
        session.status = "halted"
        session.halt_reason = f"Max turns exceeded: {session.turn_count} >= {max_turns}"
        raise CircuitBreakerTripped(session.halt_reason, session)


# 演示：主动触发熔断
demo_session = AgentSession(session_id="breaker-test", total_usd=0.48)
try:
    demo_session.total_usd += 0.05  # 推高到 0.50 美元上限之上
    check_circuit_breaker(demo_session, budget_cap=0.50)
except CircuitBreakerTripped as e:
    print(f"⚡ Circuit breaker tripped: {e.reason}")
    print(f"   Session status: {e.session.status}")

⚡ Circuit breaker tripped: Budget exceeded: $0.5300 >= $0.50
   Session status: halted


# Checkpointing：保存与恢复（Save & Restore）

每一轮结束后保存检查点，可以避免 502 错误或容器重启迫使用户从头再来。生产环境通常会把检查点写入 Redis 或 Postgres；为了让演示中的状态快照可以直接检查，这里使用 JSON 文件。

In [ ]:
CHECKPOINT_DIR = Path(tempfile.mkdtemp(prefix="agent_checkpoints_"))
print(f"Checkpoint directory: {CHECKPOINT_DIR}")


def save_checkpoint(session: AgentSession) -> Path:
    """把完整 Session 状态持久化到磁盘，并计入本次检查点。"""
    path = CHECKPOINT_DIR / f"{session.session_id}.json"
    session.checkpoints_saved += 1
    try:
        path.write_text(session.model_dump_json(indent=2))
    except Exception:
        session.checkpoints_saved -= 1
        raise
    return path


def load_checkpoint(session_id: str) -> Optional[AgentSession]:
    """从最近一次检查点恢复 Session；若不存在则返回 None。"""
    path = CHECKPOINT_DIR / f"{session_id}.json"
    if not path.exists():
        return None
    data = json.loads(path.read_text())
    return AgentSession.model_validate(data)


def list_checkpoints() -> List[str]:
    """列出所有已保存的 Session ID。"""
    return [p.stem for p in CHECKPOINT_DIR.glob("*.json")]


# 演示
demo = AgentSession(session_id="ckpt-demo", turn_count=3, total_usd=0.012)
demo.add_message("system", "You are a helpful agent.")
demo.add_message("user", "What is your return policy?")
demo.add_message("assistant", "Our return policy allows returns within 30 days.")

path = save_checkpoint(demo)
print(f"Saved checkpoint to: {path}")
print(f"Checkpoint size: {path.stat().st_size} bytes")

restored = load_checkpoint("ckpt-demo")
print(f"Restored: {restored.summary()}")
print(f"History intact: {len(restored.history)} messages")

Checkpoint directory: /tmp/agent_checkpoints_qu63q9s0
Saved checkpoint to: /tmp/agent_checkpoints_qu63q9s0/ckpt-demo.json
Checkpoint size: 551 bytes
Restored: Session ckpt-demo | status=running | turns=3 | tokens=0 | cost=$0.0120 | messages=3 | checkpoints=0
History intact: 3 messages


# 加固后的智能体循环（Hardened Agent Loop）

这里把三种模式真正组合起来。循环的每一次迭代都会：

1. 在调用模型之前**裁剪上下文**（Context Janitor）
2. **调用 LLM**，并执行任何工具调用
3. **更新指标**（token、成本、轮次计数）
4. **检查熔断器**（预算 + 轮次）
5. **保存检查点**（状态持久化）

即使任意一步失败，最后一次成功保存的检查点仍然可以用于恢复。

In [ ]:
MAX_TOOL_ROUNDS = 5  # 单轮工具调用循环的安全上限


def hardened_agent_step(
    session: AgentSession,
    user_input: str,
    budget_cap: float = 0.50,
    max_turns: int = 12,
    max_context_tokens: int = 4000,
) -> str:
    """
    执行一轮加固后的智能体流程：裁剪 → 调用 LLM → 工具 → 指标 → 熔断器 → 检查点。
    返回 assistant 的最终回复。
    """
    if session.status != "running":
        raise RuntimeError(f"Session {session.session_id} is {session.status}: {session.halt_reason}")

    # 确保系统 Prompt 已写入历史
    if not session.history or session.history[0].get("role") != "system":
        session.history.insert(0, {"role": "system", "content": session.system_prompt})

    # 添加用户消息
    session.add_message("user", user_input)

    # Context Janitor：调用模型前先裁剪上下文
    session.history = prune_history(
        session.history,
        max_tokens=max_context_tokens,
        keep_recent=6,
    )

    # LLM 调用（包含工具调用循环）
    final_reply = None
    for _round in range(MAX_TOOL_ROUNDS):
        response = client.chat.completions.create(
            model=session.model,
            messages=session.history,
            tools=TOOL_SCHEMAS,
            tool_choice="auto",
            temperature=0.2,
        )

        choice = response.choices[0]
        message = choice.message

        # 更新指标
        usage = response.usage
        prompt_tok = (usage.prompt_tokens or 0) if usage else 0
        compl_tok = (usage.completion_tokens or 0) if usage else 0
        session.total_tokens += prompt_tok + compl_tok
        session.total_usd += calculate_cost(prompt_tok, compl_tok)

        # 没有工具调用 → 得到最终答案
        tool_calls = getattr(message, "tool_calls", None)
        if not tool_calls:
            final_reply = message.content or "(no response)"
            session.add_message("assistant", final_reply)
            break

        # 执行工具调用
        session.history.append(message.model_dump())
        for tc in tool_calls:
            fn_name = tc.function.name
            try:
                fn_args = json.loads(tc.function.arguments)
            except json.JSONDecodeError:
                fn_args = {}

            try:
                result_str = execute_tool(fn_name, fn_args)
            except Exception as exc:
                result_str = json.dumps({"error": str(exc)})

            session.history.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result_str,
            })
            print(f"  🔧 Tool: {fn_name}({fn_args}) → {result_str[:80]}...")
    else:
        final_reply = "(Agent reached maximum tool-call rounds.)"
        session.add_message("assistant", final_reply)

    # 增加轮次计数
    session.turn_count += 1

    # 熔断器（Circuit Breaker）
    check_circuit_breaker(session, budget_cap=budget_cap, max_turns=max_turns)

    # 保存检查点（Checkpoint）
    ckpt_path = save_checkpoint(session)
    print(f"  💾 Checkpoint saved (turn {session.turn_count}, ${session.total_usd:.4f})")

    return final_reply


print("Hardened agent loop ready.")

Hardened agent loop ready.


---

## 演示：三种模式协同工作

下面让加固后的智能体依次通过四个场景，以验证每种机制都能真正生效：

1. **正常的多轮运行** —— 每一轮结束后保存检查点
2. **触发上下文裁剪** —— 人为膨胀历史，再观察 Context Janitor 如何裁剪
3. **触发熔断器** —— 设置很低的轮次上限，观察保险丝何时熔断
4. **崩溃恢复（Crash Recovery）** —— “杀掉”当前 Session，再从最后一个检查点恢复

### 演示 1：正常的多轮运行

这是一段接近真实的客服对话：客户有一个延迟订单，询问退货政策，然后继续追问。智能体会调用工具、每轮保存检查点，同时熔断器保持未触发状态。

In [ ]:
session1 = AgentSession(
    session_id="showcase-normal-001",
    model=MODEL,
    system_prompt=(
        "You are a customer support triage agent. Your job is to:\n"
        "1. Classify the ticket into a category (billing, technical, account, general).\n"
        "2. Assess priority (low, medium, high, urgent).\n"
        "3. Use available tools to gather context.\n"
        "4. Write a brief, empathetic response.\n"
        "Be concise. Never fabricate order details — use the lookup tool."
    ),
)

conversation = [
    "I placed order #ORD-8842 three weeks ago and nobody has told me what's going on. This is unacceptable.",
    "OK, so it's delayed. What are my options? Can I return it when it arrives if it's too late?",
    "Fine. I'll wait, but if it's not here by the 18th I want a full refund. Please escalate this.",
]

print("=" * 70)
print(" SHOWCASE 1: Normal Multi-Turn Operation")
print("=" * 70)

for i, msg in enumerate(conversation, 1):
    print(f"\n{'─' * 70}")
    print(f"Turn {i} | Customer: {msg[:80]}...")
    print(f"{'─' * 70}")
    reply = hardened_agent_step(session1, msg)
    print(f"\n  🤖 Agent: {reply[:200]}..." if len(reply) > 200 else f"\n  🤖 Agent: {reply}")

session1.status = "completed"
save_checkpoint(session1)

print(f"\n{'=' * 70}")
print(f"Final: {session1.summary()}")
print(f"{'=' * 70}")

 SHOWCASE 1: Normal Multi-Turn Operation

──────────────────────────────────────────────────────────────────────
Turn 1 | Customer: I placed order #ORD-8842 three weeks ago and nobody has told me what's going on....
──────────────────────────────────────────────────────────────────────
  🔧 Tool: classify_ticket({'summary': "I placed order #ORD-8842 three weeks ago and nobody has told me what's going on.", 'category': 'general'}) → {"category": "general", "summary": "I placed order #ORD-8842 three weeks ago and...
  🔧 Tool: lookup_order({'order_id': 'ORD-8842'}) → {"order_id": "ORD-8842", "status": "delayed", "placed": "2025-01-20", "reason": ...
  💾 Checkpoint saved (turn 1, $0.0002)

  🤖 Agent: **Category:** General  
**Priority:** Urgent  

---

Dear Customer,

I sincerely apologize for the delay with your order #ORD-8842. It is currently delayed due to a supplier back-order, and the new es...

──────────────────────────────────────────────────────────────────────
Turn 2 | Customer: 

### 演示 2：上下文裁剪生效

我们故意向历史中塞入 20 条填充消息，再发送一个真实问题。Context Janitor 应当在调用 LLM 之前清理冗余内容，同时保留系统 Prompt 和最近的对话轮次。

In [ ]:
session2 = AgentSession(
    session_id="showcase-pruning-002",
    model=MODEL,
    system_prompt="You are a customer support triage agent. Be concise.",
)

# 注入系统 Prompt
session2.history.append({"role": "system", "content": session2.system_prompt})

# 用 20 轮填充内容刻意膨胀历史
for i in range(20):
    session2.history.append({"role": "user", "content": f"Filler question {i}: " + "blah " * 60})
    session2.history.append({"role": "assistant", "content": f"Filler answer {i}: " + "response " * 60})

print("=" * 70)
print(" SHOWCASE 2: Context Pruning in Action")
print("=" * 70)
print(f"\nHistory BEFORE: {len(session2.history)} messages, {count_tokens(session2.history):,} tokens")
print(f"\nSending a real question into the bloated context...\n")

reply = hardened_agent_step(
    session2,
    "Can you look up order #ORD-5567 for me?",
    max_context_tokens=1500,  # 强制执行更激进的裁剪
)

print(f"\nHistory AFTER: {len(session2.history)} messages, {count_tokens(session2.history):,} tokens")
print(f"\n  🤖 Agent: {reply[:300]}")
print(f"\n{session2.summary()}")

 SHOWCASE 2: Context Pruning in Action

History BEFORE: 41 messages, 2,899 tokens

Sending a real question into the bloated context...

  🧹 Context Janitor: 42 msgs (2,917 tokens) → 8 msgs (432 tokens) | pruned 35 middle messages
  🔧 Tool: lookup_order({'order_id': 'ORD-5567'}) → {"order_id": "ORD-5567", "status": "shipped", "placed": "2025-02-01", "tracking"...
  💾 Checkpoint saved (turn 1, $0.0003)

History AFTER: 11 messages, 670 tokens

  🤖 Agent: Order #ORD-5567 has been shipped. Here are the details:

- **Status:** Shipped
- **Placed on:** February 1, 2025
- **Tracking Number:** 1Z999AA10123456784
- **Estimated Delivery:** February 12, 2025
- **Items:** Running Shoes (Size 10) x1

Session showcase-pruning-002 | status=running | turns=1 | tokens=1403 | cost=$0.0003 | messages=11 | checkpoints=1


### 演示 3：熔断器触发

我们把 `max_turns=3`，然后尝试发送 5 条消息。熔断器应在达到限制时触发，使 Session 进入停止状态并抛出 `CircuitBreakerTripped`。在 FastAPI 应用中，你可以捕获这个异常，并返回用户友好的“需要转人工”提示。

In [ ]:
session3 = AgentSession(
    session_id="showcase-breaker-003",
    model=MODEL,
    system_prompt="You are a customer support triage agent. Be concise.",
)

messages_to_send = [
    "What is your return policy?",
    "What about warranties?",
    "How long does shipping take?",
    "Can you look up order #ORD-1024?",  # 这条消息应触发熔断器
    "One more question...",               # 正常情况下不应执行到这里
]

print("=" * 70)
print(" SHOWCASE 3: Circuit Breaker Trips (max_turns=3)")
print("=" * 70)

for i, msg in enumerate(messages_to_send, 1):
    try:
        print(f"\n  Turn {i}: {msg}")
        reply = hardened_agent_step(session3, msg, max_turns=3)
        print(f"  🤖 {reply[:120]}..." if len(reply) > 120 else f"  🤖 {reply}")
    except CircuitBreakerTripped as e:
        print(f"\n  ⚡ CIRCUIT BREAKER TRIPPED on turn {i}!")
        print(f"     Reason: {e.reason}")
        print(f"     Session status: {e.session.status}")
        print(f"     → In production: return 'I need to hand you to a human agent.'")
        break
    except RuntimeError as e:
        print(f"\n  🚫 Session refused (already halted): {e}")
        break

print(f"\n{session3.summary()}")

 SHOWCASE 3: Circuit Breaker Trips (max_turns=3)

  Turn 1: What is your return policy?
  🔧 Tool: check_policy({'keyword': 'return'}) → {"policy": "return", "text": "Items may be returned within 30 days of delivery i...
  💾 Checkpoint saved (turn 1, $0.0001)
  🤖 Our return policy allows items to be returned within 30 days of delivery in their original packaging. Electronics must b...

  Turn 2: What about warranties?
  🔧 Tool: check_policy({'keyword': 'warranty'}) → {"policy": "warranty", "text": "All electronics carry a 1-year limited warranty ...
  💾 Checkpoint saved (turn 2, $0.0002)
  🤖 All electronics come with a 1-year limited warranty covering manufacturing defects, while accessories are covered for 90...

  Turn 3: How long does shipping take?
  🔧 Tool: check_policy({'keyword': 'shipping'}) → {"policy": "shipping", "text": "Standard shipping: 5-7 business days. Express: 2...

  ⚡ CIRCUIT BREAKER TRIPPED on turn 3!
     Reason: Max turns exceeded: 3 >= 3
     Session status: hal

### 演示 4：从检查点进行崩溃恢复

我们通过“忘掉”当前 Session 对象来模拟进程崩溃，然后从最后一个检查点恢复。智能体会从原位置继续运行：对话历史、指标和轮次计数都保持一致。

In [ ]:
print("=" * 70)
print(" SHOWCASE 4: Crash Recovery from Checkpoint")
print("=" * 70)

# 第 1 步：启动 Session，并运行 2 轮
session4 = AgentSession(
    session_id="showcase-recovery-004",
    model=MODEL,
    system_prompt="You are a customer support agent. Be concise and helpful.",
)

print("\n── Phase 1: Running 2 turns before the 'crash' ──")
reply1 = hardened_agent_step(session4, "I need to check on order #ORD-1024.")
print(f"  🤖 Turn 1: {reply1[:150]}")

reply2 = hardened_agent_step(session4, "When will it ship? I need it by Friday.")
print(f"  🤖 Turn 2: {reply2[:150]}")

print(f"\n  State before crash: {session4.summary()}")

# 第 2 步：模拟崩溃
print("\n── Phase 2: 💥 SIMULATED CRASH (deleting session object) ──")
crashed_session_id = session4.session_id
del session4  # 对象已消失：进程退出，容器重启
print(f"  Session object destroyed. Only the checkpoint remains.")

# 第 3 步：从检查点恢复
print("\n── Phase 3: Recovering from checkpoint ──")
print(f"  Available checkpoints: {list_checkpoints()}")

recovered = load_checkpoint(crashed_session_id)
if recovered:
    print(f"  ✅ Recovered: {recovered.summary()}")
    print(f"  History length: {len(recovered.history)} messages")
    print(f"  Last message: {recovered.history[-1]['content'][:100]}...")

    # 第 4 步：继续对话
    print("\n── Phase 4: Continuing the conversation after recovery ──")
    reply3 = hardened_agent_step(recovered, "Actually, can you also check the shipping policy?")
    print(f"  🤖 Turn 3 (post-recovery): {reply3[:200]}")
    print(f"\n  Final state: {recovered.summary()}")
else:
    print(f"  ❌ No checkpoint found for {crashed_session_id}")

 SHOWCASE 4: Crash Recovery from Checkpoint

── Phase 1: Running 2 turns before the 'crash' ──
  🔧 Tool: lookup_order({'order_id': 'ORD-1024'}) → {"order_id": "ORD-1024", "status": "processing", "placed": "2025-02-04", "estima...
  💾 Checkpoint saved (turn 1, $0.0001)
  🤖 Turn 1: Your order #ORD-1024 is currently processing. It was placed on February 4, 2025, and is estimated to ship by February 14, 2025. The items in your orde
  💾 Checkpoint saved (turn 2, $0.0002)
  🤖 Turn 2: Your order is estimated to ship by February 14, 2025. If you need it by Friday, please note that it may not arrive in time, as February 14 is a Wednes

  State before crash: Session showcase-recovery-004 | status=running | turns=2 | tokens=1019 | cost=$0.0002 | messages=7 | checkpoints=2

── Phase 2: 💥 SIMULATED CRASH (deleting session object) ──
  Session object destroyed. Only the checkpoint remains.

── Phase 3: Recovering from checkpoint ──
  Available checkpoints: ['showcase-recovery-004', 'showcase-breaker

---

## 汇总仪表盘（Summary Dashboard）

In [ ]:
print("=" * 80)
print(" HARDENING THE BACKBONE — SHOWCASE SUMMARY")
print("=" * 80)

# 汇总检查点中能够找到的所有 Session
all_sessions = []
for sid in list_checkpoints():
    s = load_checkpoint(sid)
    if s:
        all_sessions.append(s)

print(f"\n{'Session ID':<30} {'Status':<12} {'Turns':>6} {'Tokens':>8} {'Cost':>10} {'Checkpts':>10} {'Halt Reason'}")
print("─" * 110)
for s in sorted(all_sessions, key=lambda x: x.session_id):
    halt = s.halt_reason or "—"
    print(
        f"{s.session_id:<30} {s.status:<12} {s.turn_count:>6} "
        f"{s.total_tokens:>8} ${s.total_usd:>8.4f} {s.checkpoints_saved:>10}   {halt}"
    )

total_cost = sum(s.total_usd for s in all_sessions)
total_tokens = sum(s.total_tokens for s in all_sessions)
total_turns = sum(s.turn_count for s in all_sessions)

print(f"\n{'─' * 110}")
print(f"{'TOTAL':<30} {'':12} {total_turns:>6} {total_tokens:>8} ${total_cost:>8.4f}")

print(f"\n\n📋 Patterns Demonstrated:")
print(f"   ✅ Checkpointing  — {sum(s.checkpoints_saved for s in all_sessions)} checkpoints saved across all sessions")
print(f"   ✅ Context Pruning — Showcase 2 bloated history was trimmed before LLM call")
breaker_sessions = [s for s in all_sessions if s.status == "halted"]
print(f"   ✅ Circuit Breaker — {len(breaker_sessions)} session(s) halted by the breaker")
recovery_sessions = [s for s in all_sessions if "recovery" in s.session_id]
if recovery_sessions:
    print(f"   ✅ Crash Recovery  — Session '{recovery_sessions[0].session_id}' restored and continued")

 HARDENING THE BACKBONE — SHOWCASE SUMMARY

Session ID                     Status        Turns   Tokens       Cost   Checkpts Halt Reason
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
ckpt-demo                      running           3        0 $  0.0120          0   —
showcase-breaker-003           running           2     1310 $  0.0002          1   —
showcase-normal-001            completed         3     4055 $  0.0008          3   —
showcase-pruning-002           running           1     1403 $  0.0003          0   —
showcase-recovery-004          running           3     2062 $  0.0004          1   —

──────────────────────────────────────────────────────────────────────────────────────────────────────────────
TOTAL                                           12     8830 $  0.0137


📋 Patterns Demonstrated:
   ✅ Checkpointing  — 5 checkpoints saved across all sessions
   ✅ Context Pruning — Showcase 2 bloated history was t

## 清理（Cleanup）

In [ ]:
# 清理临时检查点目录
shutil.rmtree(CHECKPOINT_DIR, ignore_errors=True)
print(f"Cleaned up checkpoint directory: {CHECKPOINT_DIR}")

Cleaned up checkpoint directory: /tmp/agent_checkpoints_qu63q9s0
